# dlframework — Quick Start Example

This notebook shows how to use the `dlframework` micro-framework end-to-end:

1. **Feature engineering** — download OHLCV data and compute all features
2. **Training** — train a GRU model with one call to `run_trial()`
3. **Evaluation** — per-ticker metrics
4. **Recursive prediction** — multi-step forecasting with `FullRecursivePredictor`
5. **Extending** — how to add a new model, loss, or feature

---

> **Google Colab**: clone your repo, then run:
> ```python
> import sys; sys.path.insert(0, '/content/your-repo')
> ```
> before the imports below.

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yfinance as yf

warnings.filterwarnings('ignore')

# ── dlframework imports ───────────────────────────────────────────────
from dlframework import (
    # Feature engineering
    register_features, compute_all_features,
    # Constants
    FEATURE_COLS, TARGET_COL, EPS,
    # Models
    build_model, MODEL_REGISTRY,
    # Losses
    LOSS_REGISTRY,
    # Dataset
    MultiCoinDataset, load_splits,
    # Training
    run_trial, evaluate, evaluate_per_ticker,
    # Prediction
    FullRecursivePredictor, predict_non_recursive,
)
from uberdataset import UberDatasetFuhrer

DEVICE = (
    torch.device('cuda') if torch.cuda.is_available()
    else torch.device('mps') if torch.backends.mps.is_available()
    else torch.device('cpu')
)
print(f'Device: {DEVICE}')
print(f'Available models: {list(MODEL_REGISTRY.keys())}')
print(f'Available losses: {list(LOSS_REGISTRY.keys())}')

## 1. Feature Engineering

Download a few tickers, register features with `register_features()`, build them with `UberDatasetFuhrer`, and save the splits.

In [ ]:
TICKERS    = ['BTC-USD', 'ETH-USD', 'LTC-USD']
START_DATE = '2018-01-01'
DATA_DIR   = Path('./data')
DATA_DIR.mkdir(exist_ok=True)

# Download
raw = yf.download(TICKERS, start=START_DATE, interval='1d', group_by='ticker', threads=True)

# Split into per-ticker DataFrames
REQUIRED_COLS = ['Close', 'High', 'Low', 'Open', 'Volume']
ticker_frames = {}
for ticker in TICKERS:
    try:
        df = raw[ticker][REQUIRED_COLS].dropna(how='all').ffill().dropna()
        if len(df) >= 300:
            ticker_frames[ticker] = df.sort_index()
    except KeyError:
        pass

print(f'Loaded {len(ticker_frames)} tickers')

In [ ]:
# Build features for every ticker using the shared pipeline
built_frames = []

for ticker, frame in ticker_frames.items():
    ds = UberDatasetFuhrer(frame, feature_fn_prefix='ft_')
    register_features(ds)
    ds.build_all()

    if ds.df.empty:
        print(f'  {ticker}: empty after dropna, skipping')
        continue

    ds.df['ticker'] = ticker
    built_frames.append(ds.df)
    print(f'  {ticker}: {len(ds.df):,} rows')

combined_df = pd.concat(built_frames).sort_index()
print(f'\nCombined: {combined_df.shape}')

In [ ]:
# Temporal split per ticker (no shuffling to avoid look-ahead bias)
def temporal_split_per_ticker(df, train=0.8, val=0.1, test=0.1, key='_split'):
    labels = pd.Series(index=df.index, dtype='object')
    for ticker, grp in df.groupby('ticker'):
        n = len(grp)
        n_train, n_val = int(n * train), int(n * val)
        idx = grp.index
        labels.loc[idx[:n_train]]              = 'train'
        labels.loc[idx[n_train:n_train+n_val]] = 'val'
        labels.loc[idx[n_train+n_val:]]        = 'test'
    df[key] = labels
    return df

combined_df = temporal_split_per_ticker(combined_df)
print(combined_df['_split'].value_counts())

for split in ('train', 'val', 'test'):
    path = DATA_DIR / f'{split}_features.csv'
    combined_df[combined_df['_split'] == split].to_csv(path)
    print(f'{split}: saved to {path}')

## 2. Training

`load_splits()` wraps all CSV loading + DataLoader boilerplate.  
`run_trial()` handles model construction, optimizer, scheduler, early stopping, and optional W&B logging.

In [ ]:
train_loader, val_loader, test_loader, test_df = load_splits(
    DATA_DIR,
    feature_cols=FEATURE_COLS,
    target_col=TARGET_COL,
    batch_size=512,
    seq_len=20,
    device=DEVICE,
)
print(f'Train batches: {len(train_loader)}')
print(f'Val  batches:  {len(val_loader)}')
print(f'Test rows:     {len(test_df)}')

In [ ]:
# ── Training config ───────────────────────────────────────────────────
# Use a pre-registered config from MODEL_REGISTRY, or write your own dict.

cfg = {
    **MODEL_REGISTRY['MEDIUM_GRU'],   # inherit arch params
    'loss':         'huber_directional',
    'lr':           5e-4,
    'weight_decay': 1e-5,
}
print('Config:', cfg)

In [ ]:
best_val_loss, ckpt_path = run_trial(
    cfg,
    train_loader,
    val_loader,
    device=DEVICE,
    run_name='example_gru',
    checkpoint_dir='./checkpoints',
    max_epochs=30,
    patience=5,
    wandb_project=None,  # set to your project name to enable W&B logging
)
print(f'\nBest val loss: {best_val_loss:.6f}')
print(f'Checkpoint:    {ckpt_path}')

## 3. Evaluation

In [ ]:
# Load the best checkpoint
model = build_model(cfg, input_dim=len(FEATURE_COLS)).to(DEVICE)
model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
model.eval()

criterion = LOSS_REGISTRY[cfg['loss']]

# Overall test metrics
test_metrics = evaluate(model, test_loader, criterion, DEVICE)
print('── Test metrics ──────────────────────────────')
for k, v in test_metrics.items():
    print(f'  {k:<20s}: {v:.6f}')

In [ ]:
# Per-ticker breakdown
per_ticker = evaluate_per_ticker(model, test_df, FEATURE_COLS, TARGET_COL, criterion, DEVICE)

rows = []
for ticker, m in per_ticker.items():
    rows.append({'ticker': ticker, **m})

results_df = pd.DataFrame(rows).set_index('ticker')
print(results_df[['loss', 'dir_acc', 'mae', 'n']].to_string())

## 4. Recursive Prediction

`FullRecursivePredictor` automatically recomputes features at every step using the same pipeline as feature generation — no duplicated code.

In [ ]:
SELECTED_TICKER  = 'BTC-USD'
PREDICTION_DAYS  = 30
WARMUP_DAYS      = 120
SEQ_LEN          = 20
VOLUME_STRATEGY  = 'rolling_mean'   # 'repeat_last' | 'rolling_mean' | 'zero_change'
PREDICTION_SPLIT = 'val'             # 'val' | 'test'

# Load all data for the chosen ticker
all_df = pd.concat([
    pd.read_csv(DATA_DIR / 'train_features.csv', index_col=0, parse_dates=True),
    pd.read_csv(DATA_DIR / 'val_features.csv',   index_col=0, parse_dates=True),
    pd.read_csv(DATA_DIR / 'test_features.csv',  index_col=0, parse_dates=True),
])
ticker_df = all_df[all_df['ticker'] == SELECTED_TICKER].sort_index()
print(f'{SELECTED_TICKER}: {len(ticker_df)} rows  ({ticker_df.index.min()} → {ticker_df.index.max()})')

In [ ]:
# Build warmup window from the start of the chosen split
split_df = all_df[all_df['ticker'] == SELECTED_TICKER]
if PREDICTION_SPLIT == 'val':
    split_df = pd.read_csv(DATA_DIR / 'val_features.csv', index_col=0, parse_dates=True)
else:
    split_df = pd.read_csv(DATA_DIR / 'test_features.csv', index_col=0, parse_dates=True)

split_ticker = split_df[split_df['ticker'] == SELECTED_TICKER].sort_index()
split_clean  = split_ticker.dropna(subset=FEATURE_COLS + [TARGET_COL, 'target_scale'])

source_date = split_clean.index[SEQ_LEN]
source_idx  = ticker_df.index.get_loc(source_date)
if isinstance(source_idx, slice):
    source_idx = source_idx.start

warmup_start = max(0, source_idx - WARMUP_DAYS - SEQ_LEN)
warmup_df    = ticker_df.iloc[warmup_start:source_idx + 1][['Close', 'High', 'Low', 'Open', 'Volume']]
actual_df    = ticker_df.iloc[source_idx:source_idx + PREDICTION_DAYS + 1]

print(f'Warmup: {len(warmup_df)} days  ({warmup_df.index[0]} → {warmup_df.index[-1]})')
print(f'Predicting {PREDICTION_DAYS} days from {source_date}')

In [ ]:
# Non-recursive baseline (uses actual features)
dates_nr, preds_nr_scaled, actuals_nr_scaled, preds_nr, actuals_nr, scales_nr = predict_non_recursive(
    model, split_clean, FEATURE_COLS, TARGET_COL, DEVICE, SEQ_LEN
)

dir_acc_nr = np.mean(np.sign(preds_nr) == np.sign(actuals_nr))
mae_nr     = np.mean(np.abs(preds_nr - actuals_nr))
print(f'Non-recursive baseline — Dir.Acc: {dir_acc_nr:.2%}  MAE: {mae_nr:.6f}')

In [ ]:
# Full recursive prediction
predictor = FullRecursivePredictor(
    model=model,
    feature_cols=FEATURE_COLS,
    device=DEVICE,
    seq_len=SEQ_LEN,
    volume_strategy=VOLUME_STRATEGY,
)
predictor.initialize(warmup_df)

recursive_results = predictor.run_recursive_prediction(PREDICTION_DAYS)
print('Done!')

In [ ]:
# Metrics
pred_dates   = recursive_results['date'].tolist()
pred_closes  = recursive_results['close'].values
pred_returns = recursive_results['log_return_unscaled'].values

actual_closes  = [actual_df.loc[d, 'Close'] if d in actual_df.index else np.nan for d in pred_dates]
actual_returns = []
for d in pred_dates:
    if d in actual_df.index:
        idx = actual_df.index.get_loc(d)
        if idx > 0:
            prev = actual_df.index[idx - 1]
            actual_returns.append(np.log(actual_df.loc[d, 'Close'] / actual_df.loc[prev, 'Close'] + EPS))
        else:
            actual_returns.append(np.nan)
    else:
        actual_returns.append(np.nan)

actual_closes  = np.array(actual_closes)
actual_returns = np.array(actual_returns)
valid = ~np.isnan(actual_closes) & ~np.isnan(actual_returns) & np.isfinite(actual_returns)

if valid.sum() > 1:
    dir_acc_rec  = np.mean(np.sign(pred_returns[valid]) == np.sign(actual_returns[valid]))
    mae_rec      = np.mean(np.abs(pred_returns[valid] - actual_returns[valid]))
    price_err    = (pred_closes[valid][-1] - actual_closes[valid][-1]) / actual_closes[valid][-1] * 100

    print(f'Recursive      — Dir.Acc: {dir_acc_rec:.2%}  MAE: {mae_rec:.6f}  Final price error: {price_err:+.2f}%')
    print(f'Non-recursive  — Dir.Acc: {dir_acc_nr:.2%}  MAE: {mae_nr:.6f}')

In [ ]:
# Quick price chart
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(warmup_df.index[-50:], warmup_df['Close'].values[-50:],
        color='gray', alpha=0.5, label='Historical')
ax.plot([d for d in pred_dates if d in actual_df.index],
        [actual_df.loc[d, 'Close'] for d in pred_dates if d in actual_df.index],
        color='#2ecc71', lw=2, label='Actual')
ax.plot(pred_dates, pred_closes,
        color='#e74c3c', lw=2, ls='--', label='Predicted (recursive)')
ax.axvline(pred_dates[0], color='yellow', ls=':', alpha=0.7)
ax.set_title(f'{SELECTED_TICKER} — Recursive prediction ({PREDICTION_DAYS} days)')
ax.set_ylabel('Price (USD)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Extending the Framework

### Add a new model

Define your class in `dlframework/models.py`, add it to `MODEL_REGISTRY`, add an `elif` in `build_model()`. Then use it here:

In [ ]:
# Example: define a quick custom model inline (prototype before moving to models.py)
import torch.nn as nn

class TinyMLP(nn.Module):
    """Flatten the sequence and run through two linear layers."""
    def __init__(self, input_size, seq_len, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(input_size * seq_len, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)

# Test it
mlp = TinyMLP(len(FEATURE_COLS), 20).to(DEVICE)
x = torch.randn(4, 20, len(FEATURE_COLS)).to(DEVICE)
print('TinyMLP output:', mlp(x).shape)

### Add a new loss

Define in `dlframework/losses.py` and add to `LOSS_REGISTRY`. Or prototype inline:

In [ ]:
import torch.nn.functional as F

class AsymmetricLoss(nn.Module):
    """Penalise wrong-sign predictions more than correct-sign errors."""
    def __init__(self, alpha=2.0):
        super().__init__()
        self.alpha = alpha

    def forward(self, pred, target):
        base = F.huber_loss(pred, target, delta=0.1)
        wrong_sign = F.relu(-pred * target)         # > 0 only when signs differ
        return base + self.alpha * wrong_sign.mean()

# Add to registry for use in run_trial
LOSS_REGISTRY['asymmetric'] = AsymmetricLoss(alpha=2.0)

# Now you can train with:
# cfg = {**MODEL_REGISTRY['gru_basic'], 'loss': 'asymmetric', 'lr': 1e-4, 'weight_decay': 1e-5}
print('Added asymmetric loss — available losses:', list(LOSS_REGISTRY.keys()))

### Add a new feature

Add inside `register_features()` in `dlframework/features.py`, then add its z-scored column to `FEATURE_COLS`. Or test inline:

In [ ]:
# Prototype: add a 7-day close/open ratio feature
sample_ticker = list(ticker_frames.values())[0].copy()

ds_proto = UberDatasetFuhrer(sample_ticker, feature_fn_prefix='ft_')
register_features(ds_proto)

# Register the new feature on top of the existing pipeline
@ds_proto.feature(deps=['log_returns'], produces=['close_open_ratio_7'])
def ft_close_open_ratio_7(df):
    df['close_open_ratio_7'] = df['Close'].rolling(7).mean() / (df['Open'].rolling(7).mean() + EPS)

ds_proto.build_all()
print('close_open_ratio_7 sample:')
print(ds_proto.df['close_open_ratio_7'].describe())